In [ ]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XDeriveFunctor
:set -XGeneralizedNewtypeDeriving
:set -XLambdaCase
import Data.List (sortBy, nub, intercalate)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe, mapMaybe)
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: Uncertainty & Randomness"

✅ SETUP OK: Uncertainty & Randomness

# ❓ Uncertainty & Randomness in Haskell

> Категорное представление случайности и неопределённости: монады Гири, модальные логики, нечёткая математика, стохастическое программирование, битопологические пространства

| № | Тема |
|---|------|
| 1 | Неопределённость как функтор |
| 2 | Монада дискретных распределений |
| 3 | Монада Гири и вероятностные меры |
| 4 | Нечёткая математика и нечёткие множества |
| 5 | Модальные логики и фреймы Крипке |
| 6 | Стохастическое программирование: DSL для байесовских сетей |
| 7 | Марковские цепи в категории Клейсли |
| 8 | Категорная картина: стохастические матрицы и теория меры |
| 9 | Теория Пытьева → [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) |


# 1. Неопределённость как Функтор

| Тип | Семантика |
|------|----------|
| `Maybe a` | отсутствие/наличие |
| `Either e a` | ошибка/успех |
| `[a]` | множественность |
| `Validation` | накопл. ошибок |

### 🔸 эндофункторы в **Hask**

In [ ]:
-- S1: Uncertainty as Functor

data Validation e a = Failure e | Success a deriving (Show)

instance Functor (Validation e) where
  fmap _ (Failure e) = Failure e
  fmap f (Success a) = Success (f a)

instance Semigroup e => Applicative (Validation e) where
  pure = Success
  Failure e1 <*> Failure e2 = Failure (e1 <> e2)
  Failure e  <*> _          = Failure e
  _          <*> Failure e  = Failure e
  Success f  <*> Success x  = Success (f x)

safeDiv :: Int -> Int -> Maybe Int
safeDiv _ 0 = Nothing
safeDiv a b = Just (a `div` b)

parseAge :: String -> Either String Int
parseAge s = case reads s of
  [(n, "")] | n >= 0 && n <= 150 -> Right n
  [(_, "")]  -> Left ("out of range: " ++ s)
  _           -> Left ("not a number: " ++ s)

demo_sec1 :: IO ()
demo_sec1 = do
  putStrLn "\n=== S1: Uncertainty as Functor ==="
  putStrLn "\n-- Maybe:"
  print (safeDiv 10 2)
  print (safeDiv 10 0)
  putStrLn "\n-- Either:"
  print (parseAge "25")
  print (parseAge "abc")
  putStrLn "\n-- Validation:"
  let v1 = Success (+) <*> Success 1 <*> Success 2 :: Validation [String] Int
      v2 = Success (+) <*> Failure ["e1"] <*> Failure ["e2"] :: Validation [String] Int
  print v1
  print v2
  putStrLn "\n-- List nondeterminism:"
  let ms = [ (x, y) | x <- [1,2,3::Int], y <- [1,2,3] ]
  putStrLn ("3x3 = " ++ show (length ms))
  print (take 4 ms)

demo_sec1


=== S1: Uncertainty as Functor ===

-- Maybe:
Just 5
Nothing

-- Either:
Right 25
Left "not a number: abc"

-- Validation:
Success 3
Failure ["e1","e2"]

-- List nondeterminism:
3x3 = 9
[(1,1),(1,2),(1,3),(2,1)]

# 2. Монада Дискретных Распределений

`Dist a` = кон. вероят. распределение.

### 🔸 Kleisli(Dist) = **FinStoch**

In [ ]:
-- S2: Dist Monad

newtype Dist a = Dist { getProbs :: [(a, Double)] } deriving (Show)

instance Functor Dist where
  fmap f (Dist ps) = Dist [ (f a, p) | (a, p) <- ps ]

instance Applicative Dist where
  pure x = Dist [(x, 1.0)]
  Dist fs <*> Dist xs = Dist [ (f x, p*q) | (f, p) <- fs, (x, q) <- xs ]

instance Monad Dist where
  return = pure
  Dist ps >>= f = Dist [ (b, p*q) | (a, p) <- ps, (b, q) <- getProbs (f a) ]

uniform :: [a] -> Dist a
uniform xs = Dist [ (x, 1.0 / fromIntegral (length xs)) | x <- xs ]

certainly :: a -> Dist a
certainly x = Dist [(x, 1.0)]

condition :: (a -> Bool) -> Dist a -> Dist a
condition p (Dist ps) =
  let fl = [ (a, w) | (a, w) <- ps, p a ]
      t  = sum (map snd fl)
  in  Dist [ (a, w/t) | (a, w) <- fl ]

expectation :: Dist Double -> Double
expectation (Dist ps) = sum [ v * p | (v, p) <- ps ]

showDist :: Show a => Dist a -> String
showDist (Dist ps) =
  unlines [ "  " ++ show v ++ " -> " ++ show (round (p*100) :: Int) ++ "%"
           | (v, p) <- sortBy (comparing (Down . snd)) ps ]

data CoinType = Fair | Biased deriving (Show, Eq, Ord)

priorCoin :: Dist CoinType
priorCoin = Dist [(Fair, 0.5), (Biased, 0.5)]

flipCoin :: CoinType -> Dist String
flipCoin Fair   = Dist [("H", 0.5), ("T", 0.5)]
flipCoin Biased = Dist [("H", 0.9), ("T", 0.1)]

posteriorGivenH :: Dist CoinType
posteriorGivenH = condition (const True) $ do
  ct <- priorCoin
  oc <- flipCoin ct
  if oc == "H" then return ct else Dist []

demo_sec2 :: IO ()
demo_sec2 = do
  putStrLn "\n=== S2: Dist Monad ==="
  putStrLn "\n-- Uniform die:"
  putStr (showDist (uniform [1..6 :: Int]))
  let twoD = do { a <- uniform [1..6::Int]; b <- uniform [1..6::Int]; return (a+b) }
  putStrLn "\n-- Two dice:"
  putStr (showDist twoD)
  putStrLn "\n-- Bayesian coin:"
  putStr (showDist posteriorGivenH)
  let wt = Dist [("sunny", 0.6), ("rainy", 0.4)]
      md w = if w == "sunny"
               then Dist [("happy", 0.8), ("sad", 0.2)]
               else Dist [("happy", 0.3), ("sad", 0.7)]
  putStrLn "\n-- Weather->mood:"
  putStr (showDist (wt >>= md))

demo_sec2


=== S2: Dist Monad ===

-- Uniform die:
  1 -> 17%
  2 -> 17%
  3 -> 17%
  4 -> 17%
  5 -> 17%
  6 -> 17%

-- Two dice:
  2 -> 3%
  3 -> 3%
  4 -> 3%
  5 -> 3%
  6 -> 3%
  7 -> 3%
  3 -> 3%
  4 -> 3%
  5 -> 3%
  6 -> 3%
  7 -> 3%
  8 -> 3%
  4 -> 3%
  5 -> 3%
  6 -> 3%
  7 -> 3%
  8 -> 3%
  9 -> 3%
  5 -> 3%
  6 -> 3%
  7 -> 3%
  8 -> 3%
  9 -> 3%
  10 -> 3%
  6 -> 3%
  7 -> 3%
  8 -> 3%
  9 -> 3%
  10 -> 3%
  11 -> 3%
  7 -> 3%
  8 -> 3%
  9 -> 3%
  10 -> 3%
  11 -> 3%
  12 -> 3%

-- Bayesian coin:
  Biased -> 64%
  Fair -> 36%

-- Weather->mood:
  "happy" -> 48%
  "sad" -> 28%
  "sad" -> 12%
  "happy" -> 12%

# 3. Монада Гири

Markov-ядро = морфизм `X -> G(Y)` в **Kleisli(G)**.

### 🔸 `>>=` = интеграция по ядру

In [ ]:
-- S3: Giry Monad

type Kernel a b = a -> Dist b

composeK :: Kernel a b -> Kernel b c -> Kernel a c
composeK k1 k2 a = k1 a >>= k2

idKernel :: Kernel a a
idKernel = return

gaussianApprox :: Double -> Double -> Int -> Dist Double
gaussianApprox mu sigma n =
  let st  = 6 * sigma / fromIntegral n
      lo  = mu - 3 * sigma
      pts = [ lo + st * fromIntegral i | i <- [0..n-1] ]
      pdf x = exp (-(x-mu)^(2::Int) / (2*sigma^(2::Int))) / (sigma * sqrt (2*pi))
      ws  = map pdf pts
      tot = sum ws
  in  Dist (zip pts (map (/tot) ws))

betaApprox :: Double -> Double -> Int -> Dist Double
betaApprox a b n =
  let pts = [ fromIntegral i / fromIntegral n | i <- [1..n-1] ]
      pdf x = x**(a-1) * (1-x)**(b-1)
      ws  = map pdf pts
      tot = sum ws
  in  Dist (zip pts (map (/tot) ws))

rwKernel :: Double -> Kernel Double Double
rwKernel sz x = uniform [x - sz, x, x + sz]

iterateK :: Int -> Kernel a a -> Kernel a a
iterateK 0 _ = idKernel
iterateK n k = k `composeK` iterateK (n-1) k

demo_sec3 :: IO ()
demo_sec3 = do
  putStrLn "\n=== S3: Giry Monad ==="
  let g    = gaussianApprox 0 1 20
      top5 = take 5 . sortBy (comparing (Down . snd)) $ getProbs g
  putStrLn "\n-- Gaussian N(0,1):"
  mapM_ (\(x,p) -> putStrLn ("  x~" ++ show (round (x*10) :: Int)
    ++ "/10 p=" ++ show (round (p*100) :: Int) ++ "%")) top5
  let bD = betaApprox 2 5 100
  putStrLn ("\n-- Beta(2,5) mean ~ " ++ show (round (expectation bD * 1000) :: Int)
    ++ "/1000 (true: 286/1000)")
  putStrLn "\n-- RW 3 steps from 0:"
  let d3 = iterateK 3 (rwKernel 1.0) 0.0
  mapM_ (\(x,p) -> putStrLn ("  x=" ++ show x ++ " p=" ++ show (round (p*100) :: Int) ++ "%"))
        (sortBy (comparing fst) (getProbs d3))
  putStrLn "\n-- Kleisli die->coin:"
  let kD = uniform [1..6::Int]
      kC n = if even n then Dist [("H", 0.8), ("T", 0.2)]
                        else Dist [("H", 0.3), ("T", 0.7)]
  putStr (showDist (kD >>= kC))

demo_sec3


=== S3: Giry Monad ===

-- Gaussian N(0,1):
  x~0/10 p=12%
  x~3/10 p=11%
  x~-3/10 p=11%
  x~6/10 p=10%
  x~-6/10 p=10%

-- Beta(2,5) mean ~ 286/1000 (true: 286/1000)

-- RW 3 steps from 0:
  x=-3.0 p=4%
  x=-2.0 p=4%
  x=-2.0 p=4%
  x=-2.0 p=4%
  x=-1.0 p=4%
  x=-1.0 p=4%
  x=-1.0 p=4%
  x=-1.0 p=4%
  x=-1.0 p=4%
  x=-1.0 p=4%
  x=0.0 p=4%
  x=0.0 p=4%
  x=0.0 p=4%
  x=0.0 p=4%
  x=0.0 p=4%
  x=0.0 p=4%
  x=0.0 p=4%
  x=1.0 p=4%
  x=1.0 p=4%
  x=1.0 p=4%
  x=1.0 p=4%
  x=1.0 p=4%
  x=1.0 p=4%
  x=2.0 p=4%
  x=2.0 p=4%
  x=2.0 p=4%
  x=3.0 p=4%

-- Kleisli die->coin:
  "H" -> 13%
  "H" -> 13%
  "H" -> 13%
  "T" -> 12%
  "T" -> 12%
  "T" -> 12%
  "H" -> 5%
  "H" -> 5%
  "H" -> 5%
  "T" -> 3%
  "T" -> 3%
  "T" -> 3%

# 4. Нечёткая Математика

Fuzzy set = `X -> [0,1]`. Решётка `([0,1], min, max)`.

| t-норма min | `min(a,b)` | AND |
|---|---|---|
| t-норма prod | `a*b` | AND (prob) |
| t-конорма max | `max(a,b)` | OR |
| Lukasiewicz | `min(1,1-a+b)` | импл. |

### 🔸 `[0,1]` = комплектная замкн. подкат. Hask

In [ ]:
-- S4: Fuzzy Mathematics

newtype FuzzySet a = FuzzySet { fmu :: a -> Double }

tMin :: Double -> Double -> Double
tMin a b = min a b

tProd :: Double -> Double -> Double
tProd a b = a * b

tLuka :: Double -> Double -> Double
tLuka a b = max 0 (a + b - 1)

sMax :: Double -> Double -> Double
sMax a b = max a b

fuzzyNot :: Double -> Double
fuzzyNot x = 1 - x

tall :: Int -> Double
tall h
  | h < 160   = 0.0
  | h > 190   = 1.0
  | otherwise = fromIntegral (h - 160) / 30.0

young :: Int -> Double
young a
  | a < 20    = 1.0
  | a > 50    = 0.0
  | otherwise = fromIntegral (50 - a) / 30.0

lukaImpl :: Double -> Double -> Double
lukaImpl a b = min 1.0 (1 - a + b)

defuzzCentroid :: [(Double, Double)] -> Double
defuzzCentroid pts =
  let n = sum [ x * m | (x, m) <- pts ]
      d = sum [ m     | (_, m) <- pts ]
  in  if d == 0 then 0 else n / d

demo_sec4 :: IO ()
demo_sec4 = do
  putStrLn "\n=== S4: Fuzzy Mathematics ==="
  putStrLn "\n-- tall(h):"
  mapM_ (\h -> putStrLn ("  tall(" ++ show h ++ "cm) = " ++ show (tall h)))
         [155, 165, 175, 185, 195 :: Int]
  putStrLn "\n-- T-norms (0.7, 0.6):"
  putStrLn ("  min="  ++ show (tMin  0.7 0.6))
  putStrLn ("  prod=" ++ show (tProd 0.7 0.6))
  putStrLn ("  luka=" ++ show (tLuka 0.7 0.6))
  let ht = 175 :: Int
      ag = 30  :: Int
  putStrLn "\n-- h=175 age=30:"
  putStrLn ("  AND(min)  = " ++ show (tMin  (tall ht) (young ag)))
  putStrLn ("  AND(prod) = " ++ show (tProd (tall ht) (young ag)))
  putStrLn ("  L-impl(0.8->0.6) = " ++ show (lukaImpl 0.8 0.6))
  let pts = [(fromIntegral x, tall x) | x <- [160, 165..190 :: Int]]
  putStrLn ("  centroid = " ++ show (defuzzCentroid pts) ++ " cm")

demo_sec4

Line 6: Eta reduce
Found:
tMin a b = min a b
Why not:
tMin = minLine 15: Eta reduce
Found:
sMax a b = max a b
Why not:
sMax = max


=== S4: Fuzzy Mathematics ===

-- tall(h):
  tall(155cm) = 0.0
  tall(165cm) = 0.16666666666666666
  tall(175cm) = 0.5
  tall(185cm) = 0.8333333333333334
  tall(195cm) = 1.0

-- T-norms (0.7, 0.6):
  min=0.6
  prod=0.42
  luka=0.2999999999999998

-- h=175 age=30:
  AND(min)  = 0.5
  AND(prod) = 0.3333333333333333
  L-impl(0.8->0.6) = 0.7999999999999999
  centroid = 181.66666666666666 cm

# 5. Модальные Логики

Box=необходимость, Diamond=возможность. Kripke-семантика.

| S4 | `Box p->Box Box p` | транз. R |
|---|---|---|
| S5 | `Dia p->Box Dia p` | эквив. R |

### 🔸 Box = прав. сопряж. функтор

In [ ]:
-- S5: Modal Logics

data KripkeFrame w = KripkeFrame { worlds :: [w], access :: w -> [w] }

data Modal w
  = Atom    (w -> Bool)
  | MNot    (Modal w)
  | MAnd    (Modal w) (Modal w)
  | MOr     (Modal w) (Modal w)
  | MImpl   (Modal w) (Modal w)
  | Box     (Modal w)
  | Diamond (Modal w)

eval :: KripkeFrame w -> w -> Modal w -> Bool
eval _ w (Atom p)    = p w
eval f w (MNot x)    = not (eval f w x)
eval f w (MAnd x y)  = eval f w x && eval f w y
eval f w (MOr  x y)  = eval f w x || eval f w y
eval f w (MImpl x y) = not (eval f w x) || eval f w y
eval f w (Box x)     = all (\v -> eval f v x) (access f w)
eval f w (Diamond x) = any (\v -> eval f v x) (access f w)

checkAll :: Show w => KripkeFrame w -> Modal w -> [(w, Bool)]
checkAll f m = [ (w, eval f w m) | w <- worlds f ]

linearFrame :: KripkeFrame Int
linearFrame = KripkeFrame { worlds = [0..3], access = \w -> filter (>= w) [0..3] }

reflexiveFrame :: KripkeFrame Int
reflexiveFrame = KripkeFrame { worlds = [0..4], access = \w -> [w] ++ [w+1 | w < 4] }

demo_sec5 :: IO ()
demo_sec5 = do
  putStrLn "\n=== S5: Modal Logics ==="
  let ev = Atom even
      bx = Box ev
      di = Diamond ev
  putStrLn "\n-- Box(even) in S4 linear frame:"
  mapM_ (\(w,b) -> putStrLn ("  w=" ++ show w ++ " " ++ show b)) (checkAll linearFrame bx)
  putStrLn "\n-- Diamond(even):"
  mapM_ (\(w,b) -> putStrLn ("  w=" ++ show w ++ " " ++ show b)) (checkAll linearFrame di)
  putStrLn "\n-- T axiom Box(>1)->(>1) at w=2:"
  let p  = Atom (> 1)
      ax = MImpl (Box p) p
  putStrLn ("  result: " ++ show (eval reflexiveFrame 2 ax))
  putStrLn "\n-- Box(even) AND Dia(odd):"
  let f1 = MAnd (Box (Atom even)) (Diamond (Atom odd))
  mapM_ (\(w,b) -> putStrLn ("  w=" ++ show w ++ " => " ++ show b)) (checkAll linearFrame f1)

demo_sec5

Line 30: Use :
Found:
[w] ++ [w + 1 | w < 4]
Why not:
w : [w + 1 | w < 4]


=== S5: Modal Logics ===

-- Box(even) in S4 linear frame:
  w=0 False
  w=1 False
  w=2 False
  w=3 False

-- Diamond(even):
  w=0 True
  w=1 True
  w=2 True
  w=3 False

-- T axiom Box(>1)->(>1) at w=2:
  result: True

-- Box(even) AND Dia(odd):
  w=0 => False
  w=1 => False
  w=2 => False
  w=3 => False

# 6. Стохастическое Программирование: DSL

Стох. программа = морфизм в **Kleisli(Dist)**. Bayesian = disintegration.

### 🔸 Prior x Likelihood = Posterior (правило Байеса)

In [ ]:
-- S6: Stochastic DSL (Sprinkler Bayesian Network)

data Cloudy    = Cloudy    | NotCloudy    deriving (Show, Eq, Ord)
data Sprinkler = SprOn     | SprOff       deriving (Show, Eq, Ord)
data Rain      = Raining   | NotRaining   deriving (Show, Eq, Ord)
data WetGrass  = Wet       | Dry          deriving (Show, Eq, Ord)

priorCloudy :: Dist Cloudy
priorCloudy = Dist [(Cloudy, 0.5), (NotCloudy, 0.5)]

likelihoodSpr :: Cloudy -> Dist Sprinkler
likelihoodSpr Cloudy    = Dist [(SprOn, 0.1), (SprOff, 0.9)]
likelihoodSpr NotCloudy = Dist [(SprOn, 0.5), (SprOff, 0.5)]

likelihoodRain :: Cloudy -> Dist Rain
likelihoodRain Cloudy    = Dist [(Raining, 0.8), (NotRaining, 0.2)]
likelihoodRain NotCloudy = Dist [(Raining, 0.2), (NotRaining, 0.8)]

likelihoodWet :: Sprinkler -> Rain -> Dist WetGrass
likelihoodWet SprOn  Raining    = Dist [(Wet, 0.99), (Dry, 0.01)]
likelihoodWet SprOn  NotRaining = Dist [(Wet, 0.90), (Dry, 0.10)]
likelihoodWet SprOff Raining    = Dist [(Wet, 0.90), (Dry, 0.10)]
likelihoodWet SprOff NotRaining = Dist [(Wet, 0.00), (Dry, 1.00)]

jointDist :: Dist (Cloudy, Sprinkler, Rain, WetGrass)
jointDist = do
  c <- priorCloudy
  s <- likelihoodSpr c
  r <- likelihoodRain c
  w <- likelihoodWet s r
  return (c, s, r, w)

posteriorRain :: Dist Rain
posteriorRain = fmap (\(_,_,r,_) -> r) $ condition (\(_,_,_,w) -> w == Wet) jointDist

priorLinReg :: Dist (Double, Double)
priorLinReg = do
  a <- uniform [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
  b <- uniform [-2.0, -1.0, 0.0, 1.0, 2.0]
  return (a, b)

demo_sec6 :: IO ()
demo_sec6 = do
  putStrLn "\n=== S6: Stochastic Programming ==="
  putStrLn "\n-- P(Rain | WetGrass=Wet):"
  putStr (showDist posteriorRain)
  putStrLn "\n-- P(WetGrass):"
  putStr (showDist (fmap (\(_,_,_,w) -> w) jointDist))
  putStrLn "\n-- Prior regression (slope, intercept) x3:"
  mapM_ (\((a,b),p) -> putStrLn
    ("  (" ++ show a ++ ", " ++ show b ++ ") p="
     ++ show (round (p*100) :: Int) ++ "%"))
    (take 3 (getProbs priorLinReg))

demo_sec6

Line 34: Use <$>
Found:
fmap (\ (_, _, r, _) -> r)
  $ condition (\ (_, _, _, w) -> w == Wet) jointDist
Why not:
(\ (_, _, r, _) -> r)
  <$> condition (\ (_, _, _, w) -> w == Wet) jointDist


=== S6: Stochastic Programming ===

-- P(Rain | WetGrass=Wet):
  Raining -> 50%
  NotRaining -> 28%
  Raining -> 8%
  Raining -> 7%
  Raining -> 6%
  NotRaining -> 1%
  NotRaining -> 0%
  NotRaining -> 0%

-- P(WetGrass):
  Wet -> 32%
  Dry -> 20%
  Wet -> 18%
  Dry -> 9%
  Wet -> 5%
  Wet -> 5%
  Wet -> 4%
  Dry -> 4%
  Dry -> 2%
  Wet -> 1%
  Dry -> 1%
  Dry -> 0%
  Dry -> 0%
  Dry -> 0%
  Wet -> 0%
  Wet -> 0%

-- Prior regression (slope, intercept) x3:
  (-1.0, -2.0) p=3%
  (-1.0, -1.0) p=3%
  (-1.0, 0.0) p=3%

# 7. Марковские Цепи в Категории Клейсли

Markov chain = морфизм `S -> Dist S` в Kleisli(Dist).

### 🔸 Стационарное распред. = **initial algebra** для F(X)=Dist(X)

In [ ]:
-- S7: Markov Chains via Kleisli

type TransKernel s = s -> Dist s

runChain :: Int -> TransKernel s -> Dist s -> Dist s
runChain 0 _ d = d
runChain n k d = runChain (n-1) k (d >>= k)

normalizeDist :: Ord a => Dist a -> Dist a
normalizeDist (Dist ps) = Dist (Map.toList (Map.fromListWith (+) ps))

data Weather7 = Sunny7 | Cloudy7 | Rainy7 deriving (Show, Eq, Ord)

wk :: TransKernel Weather7
wk Sunny7  = Dist [(Sunny7, 0.7), (Cloudy7, 0.2), (Rainy7, 0.1)]
wk Cloudy7 = Dist [(Sunny7, 0.3), (Cloudy7, 0.4), (Rainy7, 0.3)]
wk Rainy7  = Dist [(Sunny7, 0.2), (Cloudy7, 0.3), (Rainy7, 0.5)]

gamblerK :: Int -> Dist Int
gamblerK 0 = Dist [(0, 1.0)]
gamblerK 4 = Dist [(4, 1.0)]
gamblerK n = Dist [(n-1, 0.4), (n+1, 0.6)]

demo_sec7 :: IO ()
demo_sec7 = do
  putStrLn "\n=== S7: Markov Chains ==="
  let d0 = certainly Sunny7
  mapM_ (\n -> do
    let nd = normalizeDist (runChain n wk d0)
    putStrLn ("  step " ++ show n ++ ":")
    mapM_ (\(w,p) -> putStrLn
      ("    " ++ show w ++ " = " ++ show (round (p*100) :: Int) ++ "%"))
      (getProbs nd)) [1,2,5]
  let uni  = uniform [Sunny7, Cloudy7, Rainy7]
      stat = last $ take 20 $ iterate (normalizeDist . (>>= wk)) uni
  putStrLn "\n-- Stationary distribution:"
  mapM_ (\(w,p) -> putStrLn
    ("  " ++ show w ++ " = " ++ show (round (p*1000) :: Int) ++ "/1k"))
    (getProbs stat)
  let gn = normalizeDist (runChain 5 gamblerK (certainly 2))
  putStrLn "\n-- Gambler from 2 (5 steps):"
  mapM_ (\(s,p) -> putStrLn
    ("  s=" ++ show s ++ " p=" ++ show (round (p*100) :: Int) ++ "%"))
    (getProbs gn)

demo_sec7


=== S7: Markov Chains ===
  step 1:
    Sunny7 = 70%
    Cloudy7 = 20%
    Rainy7 = 10%
  step 2:
    Sunny7 = 57%
    Cloudy7 = 25%
    Rainy7 = 18%
  step 5:
    Sunny7 = 47%
    Cloudy7 = 28%
    Rainy7 = 25%

-- Stationary distribution:
  Sunny7 = 457/1k
  Cloudy7 = 283/1k
  Rainy7 = 261/1k

-- Gambler from 2 (5 steps):
  s=0 p=24%
  s=1 p=9%
  s=3 p=14%
  s=4 p=53%

# 8. Стохастические Матрицы и Теория Меры

| Кат. | Объекты | Морф. |
|-----|----|-----|
| Hask | типы | чистые функ. |
| Kleisli(Dist) | типы | `a->Dist b` |
| FinStoch | кон. мн-ва | стох. матр |

**Монада** = (Dist, return, join). **Энтропия** — ест. инвариант.

In [ ]:
-- S8: Categorical Picture

joinDist :: Dist (Dist a) -> Dist a
joinDist (Dist outer) = Dist
  [ (x, p * q)
  | (inner, p) <- outer
  , (x, q)     <- getProbs inner
  ]

entropy :: Dist a -> Double
entropy (Dist ps) = negate (sum [ p * logBase 2 p | (_, p) <- ps, p > 0 ])

klDiv :: Ord a => Dist a -> Dist a -> Double
klDiv (Dist ps) (Dist qs) =
  sum [ p * logBase 2 (p / Map.findWithDefault 1e-10 a (Map.fromList qs))
      | (a, p) <- ps, p > 0 ]

type StochMatrix s = Map (s,s) Double

mkStoch :: Ord s => [(s, s, Double)] -> StochMatrix s
mkStoch ts = Map.fromList [ ((a,b), p) | (a,b,p) <- ts ]

applyStoch :: Ord s => StochMatrix s -> [s] -> Dist s -> Dist s
applyStoch mat ss d =
  d >>= (\s -> Dist [ (t, Map.findWithDefault 0 (s,t) mat) | t <- ss ])

demo_sec8 :: IO ()
demo_sec8 = do
  putStrLn "\n=== S8: Categorical Picture ==="
  putStrLn ("  H(die)     = " ++ show (entropy (uniform [1..6::Int])))
  putStrLn ("  H(certain) = " ++ show (entropy (certainly True)))
  putStrLn ("  H(coin)    = " ++ show (entropy (uniform [True, False])))
  let p1 = Dist [(1::Int, 0.5), (2, 0.5)]
      p2 = Dist [(1::Int, 0.8), (2, 0.2)]
  putStrLn ("  KL(fair|biased) = " ++ show (klDiv p1 p2))
  putStrLn ("  KL(biased|fair) = " ++ show (klDiv p2 p1))
  let d1 = Dist [(uniform [1,2::Int], 0.3), (uniform [3,4::Int], 0.7)]
  putStrLn ("  join: " ++ show (joinDist d1))
  let mat = mkStoch [(1,1,0.9),(1,2,0.1),(2,1,0.3),(2,2,0.7::Double)]
      nd  = normalizeDist
              (applyStoch mat [1,2] (applyStoch mat [1,2] (certainly (1::Int))))
  putStrLn "  FinStoch 2-state:"
  putStr (showDist nd)

demo_sec8


=== S8: Categorical Picture ===
  H(die)     = 2.584962500721156
  H(certain) = -0.0
  H(coin)    = 1.0
  KL(fair|biased) = 0.32192809488736235
  KL(biased|fair) = 0.27807190511263785
  join: Dist {getProbs = [(1,0.15),(2,0.15),(3,0.35),(4,0.35)]}
  FinStoch 2-state:
  1 -> 84%
  2 -> 16%

# 9. Теория возможностей Пытьева

**Теория возможностей** Ю.П. Пытьева — самостоятельная **альтернатива теории вероятностей**
для описания неопределённости. Она возникла раньше субъективного моделирования и по другим
основаниям, чем подходы Заде и Дюбуа--Прада.

Ключевая идея: меру неопределённости события задаёт не сумма вероятностей, а **супремум**
значений функции возможности. Это переводит нас из аддитивной алгебры `(+, *)` в
**решёточную** `([0,1], max, min)`.

> **Интерпретация (важно).** Теория возможностей опирается на **частотную, объективную**
> трактовку: возможность — характеристика реальности, как и вероятность. Это отличает её от
> *субъективного моделирования* (ноутбук [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)),
> где тот же математический аппарат выражает **интуицию эксперта**. Формально конструкции
> совпадают; различается источник и смысл чисел.

| Тема раздела | Суть |
|---|---|
| 9.1 Дуальные шкалы и меры | `L=([0,1],max,min)`, `L_=([0,1],min,max)`, $\Pi$ и $N$ |
| 9.2 Частотное построение | возможность из эмпирических частот |
| 9.3 Построение по вероятностям | класс $\Gamma(\mathrm{Pr})$, интервалы $\Delta_k$ |
| 9.4 Пример $w_n=1/p^n$ | одни веса — две теории |
| 9.5 «$A$ возможнее $B$» | пороговый переход $q=1$, частотный смысл |

## 9.1 Дуальные шкалы и меры возможности/необходимости

Возможность и необходимость вводятся **вместе** на двух дуальных шкалах:

$$L = ([0,1],\ \max,\ \min), \qquad \bar L = ([0,1],\ \min,\ \max).$$

Это одно и то же множество с противоположным порядком: $L \cong L^{op}$ (самодвойственность
$[0,1]$ как полной цепи). Задаются **две** функции:

- **функция возможности** $\tau: X \to [0,1]$ с нормировкой $\sup_x \tau(x) = 1$;
- **функция необходимости** $\bar\tau: X \to [0,1]$ с нормировкой $\inf_x \bar\tau(x) = 0$.

Меры события:
$$\Pi(A) = \sup_{x \in A} \tau(x), \qquad N(A) = \inf_{x \notin A} \bar\tau(x).$$

Функции $\tau$ и $\bar\tau$ можно связать **автоморфизмом дополнения** $\theta$
(строго убывающий, $\theta(0)=1$, $\theta(1)=0$): $\bar\tau = \theta\circ\tau$. В простейшем
случае $\theta(t)=1-t$. Это удобно, но **не обязательно**: $\theta$ — лишь координатное
выражение изоморфизма $L\cong L^{op}$, а не арифметическое вычитание внутри решётки. По
**принципу относительности** конкретный вид $\theta$ (и шкалы вообще) несуществен — инвариантен
лишь **порядок** значений.

| Вероятность | Теория возможностей |
|---|---|
| $\sum_x p(x)=1$ | $\sup_x \tau(x)=1$ |
| $P(A)=\sum_{x\in A}p(x)$ | $\Pi(A)=\sup_{x\in A}\tau(x)$ |
| самодвойственна | $N(A)=\inf_{x\notin A}\bar\tau(x)$ |
| $P(A\cap B)=P(A)\,P(B)$ | $\Pi(A\cap B)=\min(\Pi(A),\Pi(B))$ |

In [ ]:
-- S9.1: Теория возможностей Пытьева — дуальные шкалы и меры
-- L  = ([0,1], max, min)  — шкала возможности
-- L_ = ([0,1], min, max)  — дуальная шкала необходимости
-- tau    : X -> [0,1], sup tau  = 1   (функция возможности)
-- tauBar : X -> [0,1], inf tauBar = 0  (функция необходимости)
-- Pi(A) = sup_{x in A} tau(x),  N(A) = inf_{x not in A} tauBar(x)

data Poss a = Poss
  { pX     :: [a]            -- множество исходов X
  , pTau   :: [(a, Double)]  -- функция возможности, sup = 1
  , pTauB  :: [(a, Double)]  -- функция необходимости, inf = 0
  }

-- Координатное выражение изоморфизма L =~ L^op (необязательная связь)
theta :: Double -> Double
theta t = 1 - t

-- Построить tauBar из tau через theta (ОДИН из вариантов, не обязательный)
fromTheta :: [(a, Double)] -> [(a, Double)]
fromTheta ps = [(x, theta t) | (x, t) <- ps]

possMeasure :: Eq a => Poss a -> [a] -> Double
possMeasure m a
  | null a    = 0.0
  | otherwise = maximum [t | (x, t) <- pTau m, x `elem` a]

necMeasure :: Eq a => Poss a -> [a] -> Double
necMeasure m a =
  let compl = [x | x <- pX m, x `notElem` a]
  in if null compl then 1.0
     else minimum [tb | (x, tb) <- pTauB m, x `elem` compl]

-- Пример: X = {Sunny, Cloudy, Rainy}
data Weather = Sunny | Cloudy | Rainy deriving (Show, Eq, Ord)

wTau :: [(Weather, Double)]
wTau = [(Sunny, 1.0), (Cloudy, 0.6), (Rainy, 0.3)]   -- sup = 1

weather :: Poss Weather
weather = Poss [Sunny, Cloudy, Rainy] wTau (fromTheta wTau)

demo_s91 :: IO ()
demo_s91 = do
  putStrLn "=== 9.1 Дуальные меры Pi и N ==="
  let a = [Sunny, Cloudy]
  putStrLn $ "tau:    " ++ show wTau
  putStrLn $ "tauBar: " ++ show (fromTheta wTau)
  putStrLn $ "Pi({Sunny,Cloudy}) = sup tau   = " ++ show (possMeasure weather a)
  putStrLn $ "N ({Sunny,Cloudy}) = inf tauBar(compl) = " ++ show (necMeasure weather a)
  putStrLn $ "Pi(X) = " ++ show (possMeasure weather [Sunny,Cloudy,Rainy]) ++ " (нормировка sup=1)"

demo_s91

=== 9.1 Дуальные меры Pi и N ===
tau:    [(Sunny,1.0),(Cloudy,0.6),(Rainy,0.3)]
tauBar: [(Sunny,0.0),(Cloudy,0.4),(Rainy,0.7)]
Pi({Sunny,Cloudy}) = sup tau   = 1.0
N ({Sunny,Cloudy}) = inf tauBar(compl) = 0.7
Pi(X) = 1.0 (нормировка sup=1)

## 9.2 Частотное построение возможностной модели

Объективная (частотная) трактовка позволяет строить $\tau$ **прямо из опыта**. Пусть в серии
из $n$ независимых опытов исход $x$ встретился $\nu_n(x)$ раз. Нормируем не суммой, а
**супремумом**:
$$\tau_n(x) = \frac{\nu_n(x)}{\max_y \nu_n(y)}, \qquad \sup_x \tau_n(x) = 1.$$

Самый частый исход получает возможность $1$, остальные — в долях от него. При $n\to\infty$
последовательность $\tau_n$ стабилизируется и даёт объективную функцию возможности $\tau$.

В отличие от вероятности, здесь **не требуется** суммируемость частот: нормировка супремумом
корректна всегда. Именно это делает теорию возможностей применимой там, где вероятностная
нормировка невозможна (см. «жирные хвосты» в 9.4).

In [ ]:
-- S9.2: Частотное построение возможностной модели
-- Наблюдаем серию опытов; nu(x) — число появлений x.
-- Нормировка супремумом: tau(x) = nu(x) / max_y nu(y).

possFromFreq :: [(a, Int)] -> [(a, Double)]
possFromFreq counts =
  let mx = maximum (map snd counts)
  in if mx == 0
     then [(x, 0.0) | (x, _) <- counts]
     else [(x, fromIntegral c / fromIntegral mx) | (x, c) <- counts]

-- Для сравнения: частотная ВЕРОЯТНОСТЬ (нормировка суммой)
probFromFreq :: [(a, Int)] -> [(a, Double)]
probFromFreq counts =
  let s = sum (map snd counts)
  in [(x, fromIntegral c / fromIntegral s) | (x, c) <- counts]

demo_s92 :: IO ()
demo_s92 = do
  putStrLn "=== 9.2 Частотное построение ==="
  -- 1000 опытов: A=600, B=300, C=100
  let counts = [("A", 600 :: Int), ("B", 300), ("C", 100)]
  putStrLn $ "Частоты nu: " ++ show counts
  putStrLn $ "Вероятность p(x) = nu/sum:  " ++ show (probFromFreq counts)
  putStrLn $ "Возможность tau(x) = nu/max: " ++ show (possFromFreq counts)
  putStrLn $ "  sup tau = " ++ show (maximum (map snd (possFromFreq counts))) ++ " (норм. выполнена)"
  putStrLn "Самый частый исход получает возможность 1; остальные — его доли."

demo_s92

=== 9.2 Частотное построение ===
Частоты nu: [("A",600),("B",300),("C",100)]
Вероятность p(x) = nu/sum:  [("A",0.6),("B",0.3),("C",0.1)]
Возможность tau(x) = nu/max: [("A",1.0),("B",0.5),("C",0.16666666666666666)]
  sup tau = 1.0 (норм. выполнена)
Самый частый исход получает возможность 1; остальные — его доли.

## 9.3 Построение возможностной модели по вероятностям

Если вероятность $\mathrm{Pr}$ уже известна, согласованную с ней возможность выбирают из
**класса** $\Gamma(\mathrm{Pr})$ — функций вида $\tau=\gamma\circ\mathrm{Pr}$ со строго
возрастающим $\gamma$. «Согласована» значит: **порядок** возможностей не противоречит порядку
вероятностей.

Каноническое (Дюбуа--Прада / Пытьев) построение: упорядочим
$\mathrm{pr}_1\ge\mathrm{pr}_2\ge\cdots$ и возьмём **хвостовые суммы**
$$\tau_k = \sum_{j\ge k}\mathrm{pr}_j, \qquad \tau_1=1,\ \tau_1\ge\tau_2\ge\cdots$$

Свобода выбора $\gamma$ описывается **интервалами допустимых вероятностей**:
$$\Delta_k = \bigl[\mathrm{pr}_k,\ 1-\mathrm{pr}_1-\cdots-\mathrm{pr}_{k-1}\bigr].$$

Когда интервалы $\Delta_k$ разделены, возможность нетривиальна; когда смыкаются — возможность
вырождается в тривиальную ($\tau_k\equiv1$). Этот порог детально разобран в 9.5.

In [ ]:
-- S9.3: Построение возможностной модели по вероятностям
import Data.List (sort, sortBy)
import Data.Ord (comparing, Down(..))

-- Каноническая возможность из вероятностей: хвостовые суммы
-- tau_k = sum_{j>=k} pr_j  при pr_1 >= pr_2 >= ...
possFromProb :: [Double] -> [Double]
possFromProb prs =
  let sorted = sortBy (comparing Down) prs   -- pr_1 >= pr_2 >= ...
  in [sum (drop (k-1) sorted) | k <- [1 .. length sorted]]

-- Интервалы допустимых вероятностей Delta_k
deltaK :: [Double] -> Int -> (Double, Double)
deltaK prs k =
  let sorted = sortBy (comparing Down) prs
      lo = sorted !! (k-1)
      hi = 1.0 - sum (take (k-1) sorted)
  in (lo, hi)

-- Пересекаются ли соседние интервалы (смыкание => тривиальная возможность)
adjacentSeparated :: [Double] -> Bool
adjacentSeparated prs =
  let n  = length prs
      ds = map (deltaK prs) [1 .. n]
      gap (_, hi) (lo, _) = lo > hi    -- зазор между Delta_{k+1} и Delta_k
  in and (zipWith gap (tail ds) ds)

demo_s93 :: IO ()
demo_s93 = do
  putStrLn "=== 9.3 Возможность по вероятностям ==="
  -- Случай 1: p=2 (q=1), pr_k = 1/2^k — интервалы смыкаются
  let pr2 = [0.5, 0.25, 0.125, 0.0625, 0.0625]
  putStrLn $ "Pr (p=2): " ++ show pr2
  putStrLn $ "tau (хвостовые суммы): " ++ show (possFromProb pr2)
  putStrLn $ "Delta_k: " ++ show (map (deltaK pr2) [1..5])
  putStrLn $ "Интервалы разделены? " ++ show (adjacentSeparated pr2) ++ " => возможность тривиальна"
  putStrLn ""
  -- Случай 2: p=3 (q=2), pr_k = 2/3^k — интервалы разделены
  let pr3 = [0.6667, 0.2222, 0.0741, 0.0247, 0.0123]
  putStrLn $ "Pr (p=3): " ++ show pr3
  putStrLn $ "tau (хвостовые суммы): " ++ show (possFromProb pr3)
  putStrLn $ "Интервалы разделены? " ++ show (adjacentSeparated pr3) ++ " => возможность нетривиальна"

demo_s93

=== 9.3 Возможность по вероятностям ===
Pr (p=2): [0.5,0.25,0.125,6.25e-2,6.25e-2]
tau (хвостовые суммы): [1.0,0.5,0.25,0.125,6.25e-2]
Delta_k: [(0.5,1.0),(0.25,0.5),(0.125,0.25),(6.25e-2,0.125),(6.25e-2,6.25e-2)]
Интервалы разделены? False => возможность тривиальна

Pr (p=3): [0.6667,0.2222,7.41e-2,2.47e-2,1.23e-2]
tau (хвостовые суммы): [1.0,0.3333,0.1111,3.7e-2,1.23e-2]
Интервалы разделены? True => возможность нетривиальна

## 9.4 Пример: $w_n = 1/p^n$ — одни веса, две теории

Пытьев использует следующий пример для иллюстрации принципиального различия
между теорией вероятностей и теорией возможностей.

**Постановка.** Пусть $X = \{0, 1, 2, \ldots\}$ — счётное множество элементарных событий,
$p > 1$ — фиксированный параметр. Задан единственный набор весов:
$$w_n = \frac{1}{p^n}, \quad n = 0, 1, 2, \ldots$$

Те же числа $w_n$ порождают две совершенно разные теории.

### Теория вероятностей: нормировка суммой

Для вероятностной меры необходимо $\sum_n p(n) = 1$. Поскольку
$$\sum_{n=0}^{\infty} w_n = \frac{p}{p-1},$$
нормированное распределение:
$$p(n) = \frac{w_n}{\sum_k w_k} = \frac{p-1}{p} \cdot \frac{1}{p^n} = \left(1-\frac{1}{p}\right)\left(\frac{1}{p}\right)^n$$
— это **геометрическое распределение** с параметром $q = 1/p$.

Основные характеристики:
$$\mathbf{E}[N] = \frac{1}{p-1}, \quad P(N \geq k) = \left(\frac{1}{p}\right)^k.$$

### Теория возможностей (Пытьев): нормировка супремумом

В теории возможностей нормировка производится через **супремум**:
$$\tau(n) = w_n = \frac{1}{p^n}, \qquad \sup_n \tau(n) = \tau(0) = 1 \checkmark$$

Никакого перемасштабирования не нужно! Мера возможности:
$$\Pi(A) = \sup_{n \in A} \tau(n) = \frac{1}{p^{\min A}}, \qquad \Pi(N \geq k) = \frac{1}{p^k}.$$

### Числа совпадают, алгебра различается

**Хвостовые события** дают одинаковые числа:
$$\Pi(N \geq k) = P(N \geq k) = \frac{1}{p^k}.$$

Однако **одноточечные вероятности** в сумме дают 1 (сигма-аддитивность),
тогда как одноточечные возможности в сумме дают:
$$\sum_{n=0}^{\infty} \Pi(\{n\}) = \sum_{n=0}^{\infty} \frac{1}{p^n} = \frac{p}{p-1} > 1.$$

Это не дефект, а **принципиальное свойство теории возможностей**: мера $\Pi$
максимально аддитивна $(\Pi(A \cup B) = \max(\Pi(A),\Pi(B)))$, а не сигма-аддитивна.

### Интеграл Пытьева–Сугено vs математическое ожидание

Для $f(n) = n$ (значение исхода):

$$\mathbf{E}[N] = \sum_n n \cdot p(n) = \frac{1}{p-1}$$

$$\mathrm{pl}(N) = \sup_n \min\!\left\{\tau(n),\, n\right\} = \sup_n \min\!\left\{\frac{1}{p^n},\, n\right\} = \frac{1}{p}$$

(максимум достигается при $n=1$: $\min(1/p,\, 1) = 1/p$; при $n \geq 2$: $1/p^n < 1/p$).

### «Жирные хвосты»: невозможное вероятностное, возможное — возможностное

| Веса $w_n$ | Вероятность | Теория возможностей |
|------------|-------------|---------------------|
| $1/p^n$, $p>1$ | Геометр. распр., $\mathbf{E}[N]=1/(p-1)$ | $\tau(n)=1/p^n$, $\sup=1$, уже нормировано |
| $1/n$, $n\geq 1$ | **Невозможно**: $\sum 1/n = \infty$ | $\tau(n)=1/n$, $\sup=\tau(1)=1$ ✓ |
| $1/\sqrt{n}$ | **Невозможно**: расходится | ✓ валидно |
| $n^{-\alpha}$, $0<\alpha\leq 1$ | **Невозможно**: $\zeta(\alpha)=\infty$ | ✓ валидно всегда |

Для вероятностной теории требуется $\sum w_n < \infty$; теория возможностей
требует лишь $\sup w_n < \infty$ — всегда выполнено для ограниченных последовательностей.

![Вероятность vs Возможность: w_n = 1/p^n](../diagrams/misc/poss_vs_prob.svg)

In [ ]:
-- S9.4: Теория вероятностей vs Теория возможностей (Пытьев)
-- Пример: X = {0,1,2,...}, w_n = 1/p^n

-- Веса
wn :: Double -> Int -> Double
wn p n = 1.0 / p ^ n

-- Вероятностная теория: нормировка суммой (геометрическое распределение)
-- p(n) = (1 - 1/p) * (1/p)^n
probGeom :: Double -> Int -> Double
probGeom p n = (1.0 - 1.0/p) * (1.0/p) ^ n

-- Теория возможностей: нормировка супремумом (уже нормировано при n=0)
tauPoss :: Double -> Int -> Double
tauPoss p n = wn p n   -- sup = tau(0) = 1, перемасштабирование не нужно

-- Вероятность хвостового события P(N >= k)
probTail :: Double -> Int -> Double
probTail p k = (1.0/p) ^ k

-- Возможность хвостового события Pi(N >= k) = sup_{n>=k} tau(n) = tau(k)
possTail :: Double -> Int -> Double
possTail p k = tauPoss p k   -- = 1/p^k — численно то же, что probTail!

-- Сумма одноточечных вероятностей (должна = 1)
probSumN :: Double -> Int -> Double
probSumN p n = sum [probGeom p k | k <- [0..n]]

-- Сумма одноточечных возможностей (стремится к p/(p-1) > 1)
possSumN :: Double -> Int -> Double
possSumN p n = sum [tauPoss p k | k <- [0..n]]

-- Математическое ожидание E[N] = 1/(p-1)  (точное значение)
expectedValue :: Double -> Double
expectedValue p = 1.0 / (p - 1.0)

-- Интеграл Пытьева-Сугено pl(f) = sup_n min(tau(n), f(n)) для f(n)=n
-- Максимум при n=1: min(1/p, 1) = 1/p
plIntegralId :: Double -> Int -> Double
plIntegralId p maxN = maximum [min (tauPoss p n) (fromIntegral n) | n <- [0..maxN]]

-- Гармонические веса: tau(n) = 1/n -- ВАЛИДНО для теории возможностей
tauHarmonic :: Int -> Double
tauHarmonic 0 = 0.0        -- условно (0 не входит в X)
tauHarmonic n = 1.0 / fromIntegral n
-- sup_{n>=1} tauHarmonic n = tauHarmonic 1 = 1.0  -- нормировка выполнена

harmonicSum :: Int -> Double
harmonicSum n = sum [1.0 / fromIntegral k | k <- [1..n]]

demo_poss_prob :: IO ()
demo_poss_prob = do
  let p = 2.0 :: Double
  putStrLn "=== Пример: w_n = 1/p^n, p = 2 ==="
  putStrLn ""

  putStrLn "--- Нормировка ---"
  putStrLn $ "Сумма весов sum(w_n, n=0..20) = " ++ show (sum [wn p k | k <- [0..20]])
  putStrLn $ "Теор. значение p/(p-1) = " ++ show (p / (p-1))
  putStrLn ""

  putStrLn "--- Хвостовые события (числа совпадают!) ---"
  mapM_ (\k -> putStrLn $
    "  k=" ++ show k ++
    ":  P(N>=k) = " ++ show (probTail p k) ++
    "  Pi(N>=k) = " ++ show (possTail p k))
    [0..5 :: Int]
  putStrLn ""

  putStrLn "--- Одноточечные меры (числа РАЗЛИЧАЮТСЯ) ---"
  putStrLn $ "Сумма p({n}, n=0..20):   " ++ show (probSumN p 20) ++ " --> 1"
  putStrLn $ "Сумма Pi({n}, n=0..20):  " ++ show (possSumN p 20) ++ " --> " ++ show (p/(p-1))
  putStrLn ""

  putStrLn "--- Интегралы ---"
  putStrLn $ "E[N] = 1/(p-1)          = " ++ show (expectedValue p)
  putStrLn $ "pl(N) = sup min(tau,n)  = " ++ show (plIntegralId p 50)
  putStrLn $ "  (максимум при n=1: min(1/2, 1) = 0.5)"
  putStrLn ""

  putStrLn "--- Гармонические веса: tau(n) = 1/n ---"
  putStrLn $ "sup tau(n) = tau(1) = " ++ show (tauHarmonic 1) ++ " -- ВАЛИДНАЯ теория возможностей"
  putStrLn $ "sum 1/n для n=1..1000 = " ++ show (harmonicSum 1000)
  putStrLn "  --> расходится: НЕ может быть вероятностным распределением"

demo_poss_prob

Line 15: Eta reduce
Found:
tauPoss p n = wn p n
Why not:
tauPoss = wnLine 23: Eta reduce
Found:
possTail p k = tauPoss p k
Why not:
possTail = tauPossLine 51: Use camelCase
Found:
demo_poss_prob :: IO ()
Why not:
demoPossProb :: IO ()Line 52: Use camelCase
Found:
demo_poss_prob = ...
Why not:
demoPossProb = ...Line 78: Redundant $
Found:
putStrLn $ "  (максимум при n=1: min(1/2, 1) = 0.5)"
Why not:
putStrLn "  (максимум при n=1: min(1/2, 1) = 0.5)"

=== Пример: w_n = 1/p^n, p = 2 ===

--- Нормировка ---
Сумма весов sum(w_n, n=0..20) = 1.9999990463256836
Теор. значение p/(p-1) = 2.0

--- Хвостовые события (числа совпадают!) ---
  k=0:  P(N>=k) = 1.0  Pi(N>=k) = 1.0
  k=1:  P(N>=k) = 0.5  Pi(N>=k) = 0.5
  k=2:  P(N>=k) = 0.25  Pi(N>=k) = 0.25
  k=3:  P(N>=k) = 0.125  Pi(N>=k) = 0.125
  k=4:  P(N>=k) = 6.25e-2  Pi(N>=k) = 6.25e-2
  k=5:  P(N>=k) = 3.125e-2  Pi(N>=k) = 3.125e-2

--- Одноточечные меры (числа РАЗЛИЧАЮТСЯ) ---
Сумма p({n}, n=0..20):   0.9999995231628418 --> 1
Сумма Pi({n}, n=0..20):  1.9999990463256836 --> 2.0

--- Интегралы ---
E[N] = 1/(p-1)          = 1.0
pl(N) = sup min(tau,n)  = 0.5
  (максимум при n=1: min(1/2, 1) = 0.5)

--- Гармонические веса: tau(n) = 1/n ---
sup tau(n) = tau(1) = 1.0 -- ВАЛИДНАЯ теория возможностей
sum 1/n для n=1..1000 = 7.485470860550343
  --> расходится: НЕ может быть вероятностным распределением

## 9.5 Что значит «одно событие возможнее другого» с точки зрения вероятности

*(По: Пытьев Ю.П., Шишмарев И.А., 2010, разд. 3.2.3)*

### Постановка вопроса

Пусть задано вероятностное пространство $(\Omega, \mathcal{P}(\Omega), \mathrm{Pr})$, где
$\Omega = \{\omega_1, \omega_2, \ldots\}$ счётно. Как выбрать функцию возможностей
$\tau: \Omega \to [0,1]$, **согласованную** с данной вероятностью?

Пытьев требует, чтобы $\tau$ принадлежала **классу** $\Gamma(\mathrm{Pr})$ — множеству всех
функций вида $\tau = \gamma \circ \mathrm{Pr}$ для допустимых $\gamma \in \Gamma$, строго
монотонно возрастающих на $[0,1]$. Говоря неформально: «возможность согласована с
вероятностью, если порядок возможностей не противоречит порядку вероятностей».

Ключевую роль играют **интервалы допустимых вероятностей**:
$$\Delta_k = \bigl[\mathrm{Pr}(\{\omega_k\}),\ 1 - \mathrm{Pr}(\{\omega_1\}) - \cdots - \mathrm{Pr}(\{\omega_{k-1}\})\bigr]$$

### Случай 1 — $q = 1$ ($p = 2$, $w_n = 1/2^n$): тривиальная возможность

Для геометрического распределения $\mathrm{pr}_k = 1/2^k$:

$$\Delta_1 = [1/2,\ 1], \quad \Delta_2 = [1/4,\ 1/2], \quad \Delta_3 = [1/8,\ 1/4], \ldots$$

Интервалы **плотно упакованы** и **не перекрываются** — но вот что важно: функция
$\gamma \in \Gamma$ должна быть **константной на каждом $\Delta_k$** (значение $\gamma$
фиксировано классом допустимости). Поскольку все интервалы «сцеплены» в единую цепочку,
прикрывающую $(0, 1]$, единственная согласованная функция:

$$\tau_k = \gamma\bigl(\mathrm{pr}_k\bigr) \equiv 1 \quad \forall k.$$

**Все исходы равновозможны**, хотя их вероятности убывают в геометрической прогрессии:

$$\mathrm{Pr}(\{\omega_1\}) = \frac{1}{2} \gg \mathrm{Pr}(\{\omega_{100}\}) = \frac{1}{2^{100}} \gg \cdots$$

> **Смысл:** вероятностные градации $1/2,\ 1/4,\ 1/8, \ldots$ «слишком плотны», чтобы
> порождать нетривиальный порядок возможностей. Видя только частоты (которые различаются
> в $2^k$ раз!), модельер **не может** утверждать, что $\omega_1$ «возможнее» $\omega_2$.

### Случай 2 — $q > 1$ ($p > 2$, $w_n = 1/p^n$): нетривиальная возможность

Для $\mathrm{pr}_k = (p-1)/p^k$ при $p > 2$: интервалы $\Delta_k$ **разделяются**
(не пересекаются и между ними есть зазор). Тогда $\Gamma(\mathrm{Pr})$ нетривиален, и
существует **строго убывающее** распределение возможностей:

$$\tau_1 = 1 > \tau_2 > \tau_3 > \cdots > 0.$$

При этом возникает **потрясающее явление** — одиночество возможнее хвоста:

$$\Pi(\{\omega_k\}) = \tau_k > \tau_{k+1} = \Pi(\{\omega_{k+1}, \omega_{k+2}, \ldots\}),$$

хотя в вероятностном смысле:
$$\mathrm{Pr}(\{\omega_k\}) = \frac{p-1}{p^k}, \quad \mathrm{Pr}(\{\omega_{k+1}, \ldots\}) = \frac{1}{p^k}.$$

Для $p > 2$: $\mathrm{Pr}(\{\omega_k\}) / \mathrm{Pr}(\{\omega_{k+1},\ldots\}) = p-1 > 1$,
т.е. одиночество и **вероятнее** хвоста. Порядки **совпадают**.

Но сравните по-другому: $\{\omega_k\}$ vs $\{\omega_1, \ldots, \omega_{k-1}\}$ (все предшественники):
$$\Pi(\{\omega_k\}) = \tau_k < \tau_1 = \Pi(\{\omega_1, \ldots, \omega_{k-1}\}),$$
$$\mathrm{Pr}(\{\omega_k\}) \ll \mathrm{Pr}(\{\omega_1,\ldots,\omega_{k-1}\}).$$
Оба порядка совпадают. Но:

### Истинная красота: пороговый переход $q = 1$

| Параметр | Вероятности $\mathrm{pr}_k$ | Возможности $\tau_k$ | Различимы ли? |
|----------|----------------------------|----------------------|---------------|
| $q < 1$ ($p < 2$) | $\mathrm{pr}_k$ убывают слабо | тривиальны ($\tau_k \equiv 1$) | Нет |
| $q = 1$ ($p = 2$) | $\mathrm{pr}_k = 1/2^k$ | тривиальны ($\tau_k \equiv 1$) | **Нет!** |
| $q > 1$ ($p > 2$) | $\mathrm{pr}_k$ убывают быстро | нетривиальны, убывают | **Да** |

При $q = 1$: $\omega_1$ в $2^{999999}$ раз вероятнее $\omega_{10^6}$, но **одинаково возможны**.
При $q > 1$: возникает строгий порядок $\tau_1 > \tau_2 > \ldots$, согласованный с частотами.

### Что значит $\Pi(A) > \Pi(B)$: частотная интерпретация

*Теорема (Пытьев, 2010):* $\Pi(A) > \Pi(B)$ тогда и только тогда, когда для
**любой последовательности опытов с меняющимися вероятностями**, сохраняющей порядок
$\mathrm{Pr}(A) > \mathrm{Pr}(B)$, выполняется:

> «В достаточно длинной серии $n$ независимых опытов частота $A$ превысит частоту $B$
> почти наверное для всех $n > N(A,B)$.»

Иными словами: **«возможнее» = «будет встречаться чаще при любом варианте вероятностей,
сохраняющем порядок»**. Это более робастное утверждение, чем «вероятнее при данных $\mathrm{Pr}$».

In [ ]:
-- S9.5: Что значит «одно событие возможнее другого»
-- Пример: геометрическое распределение, q = 1 vs q > 1

-- Вероятности: pr_k = q/(1+q)^k, k = 1,2,...
probGeo :: Double -> Int -> Double
probGeo q k = q / (1+q)^k

-- Интервалы Delta_k = [pr_k, 1 - pr_1 - ... - pr_{k-1}]
delta :: Double -> Int -> (Double, Double)
delta q k =
  let lo = probGeo q k
      hi = 1.0 - sum [probGeo q j | j <- [1..k-1]]
  in (lo, hi)

-- Пересекаются ли два интервала?
overlaps :: (Double,Double) -> (Double,Double) -> Bool
overlaps (a1,b1) (a2,b2) = a1 < b2 && a2 < b1

-- Для q=1 (p=2): проверим, что интервалы смыкаются (нет зазора)
-- hi(k) = 1 - sum_{j=1}^{k-1} 1/2^j = 1 - (1 - 1/2^{k-1}) = 1/2^{k-1}
-- lo(k) = 1/2^k
-- hi(k+1) = 1/2^k = lo(k) => интервалы смыкаются точно, без зазора!

-- Возможность (Пытьев): для q<=1 она тривиальна (все = 1)
--                       для q>1  она нетривиальна (строго убывает)
possibilityTrivial :: Double -> Bool
possibilityTrivial q = q <= 1.0

-- Для нетривиального случая q>1: tau_k пропорционально delta
-- (в точности: tau_k = p_k из класса Gamma(Pr), здесь упрощённо)
-- Используем tau_k = 1/p^(k-1) как репрезентант (p = 1+q)
tauNonTrivial :: Double -> Int -> Double
tauNonTrivial q k =
  let p = 1 + q
  in 1.0 / p^(k-1)   -- tau_1 = 1, tau_2 = 1/p, tau_3 = 1/p^2, ...

-- Мера возможности множества {k}
possEventSingleton :: Double -> Int -> Double
possEventSingleton q k
  | q <= 1.0  = 1.0                      -- тривиальная возможность
  | otherwise = tauNonTrivial q k

-- Мера возможности хвоста {k+1, k+2, ...}
possEventTail :: Double -> Int -> Double
possEventTail q k
  | q <= 1.0  = 1.0
  | otherwise = tauNonTrivial q (k+1)   -- sup = следующий элемент

demo_meaning :: IO ()
demo_meaning = do
  putStrLn "=== Что значит 'A возможнее B': случай q=1 vs q=2 ==="
  putStrLn ""

  -- q=1: тривиальная возможность
  let q1 = 1.0
  putStrLn $ "--- q = " ++ show q1 ++ " (p=2, w_k = 1/2^k) ---"
  putStrLn "Вероятности убывают в 2^k раз, но возможности:"
  mapM_ (\k -> putStrLn $
    "  Pr({w_" ++ show k ++ "}) = " ++ show (probGeo q1 k) ++
    "   Pi({w_" ++ show k ++ "}) = " ++ show (possEventSingleton q1 k))
    [1..5 :: Int]
  putStrLn "=> Все исходы РАВНОВОЗМОЖНЫ! Pi({w_1}) = Pi({w_million}) = 1"
  putStrLn ""

  -- q=2: нетривиальная возможность
  let q2 = 2.0
  putStrLn $ "--- q = " ++ show q2 ++ " (p=3, w_k = 2/3^k) ---"
  putStrLn "Вероятности и возможности:"
  mapM_ (\k -> do
    let pr_k  = probGeo q2 k
        pi_k  = possEventSingleton q2 k
        pi_tl = possEventTail q2 k
        pr_tl = 1.0 - sum [probGeo q2 j | j <- [1..k]]
    putStrLn $
      "  k=" ++ show k ++
      ": Pr={k}=" ++ show (round (pr_k*1000) `div` 1 :: Int) ++ "/1000" ++
      "  Pi={k}=" ++ show pi_k ++
      "  Pi(tail)=" ++ show pi_tl ++
      "  Pr(tail)=" ++ show (round (pr_tl*1000) `div` 1 :: Int) ++ "/1000")
    [1..4 :: Int]
  putStrLn ""
  putStrLn "Вывод: {k} возможнее хвоста {k+1,...} при q>1,"
  putStrLn "хотя вероятности тоже упорядочены (Pr({k}) > Pr(хвоста) для q>1)."
  putStrLn ""

  -- Пороговый переход
  putStrLn "--- Пороговый переход: когда возможность становится нетривиальной? ---"
  mapM_ (\q ->
    putStrLn $ "  q=" ++ show q ++
      ": тривиальная=" ++ show (possibilityTrivial q) ++
      "  Pi({1}) vs Pi({2}) = " ++ show (possEventSingleton q 1) ++ " vs " ++ show (possEventSingleton q 2))
    [0.5, 0.9, 1.0, 1.1, 2.0, 5.0 :: Double]

demo_meaning

Line 49: Use camelCase
Found:
demo_meaning :: IO ()
Why not:
demoMeaning :: IO ()Line 50: Use camelCase
Found:
demo_meaning = ...
Why not:
demoMeaning = ...

=== Что значит 'A возможнее B': случай q=1 vs q=2 ===

--- q = 1.0 (p=2, w_k = 1/2^k) ---
Вероятности убывают в 2^k раз, но возможности:
  Pr({w_1}) = 0.5   Pi({w_1}) = 1.0
  Pr({w_2}) = 0.25   Pi({w_2}) = 1.0
  Pr({w_3}) = 0.125   Pi({w_3}) = 1.0
  Pr({w_4}) = 6.25e-2   Pi({w_4}) = 1.0
  Pr({w_5}) = 3.125e-2   Pi({w_5}) = 1.0
=> Все исходы РАВНОВОЗМОЖНЫ! Pi({w_1}) = Pi({w_million}) = 1

--- q = 2.0 (p=3, w_k = 2/3^k) ---
Вероятности и возможности:
  k=1: Pr={k}=667/1000  Pi={k}=1.0  Pi(tail)=0.3333333333333333  Pr(tail)=333/1000
  k=2: Pr={k}=222/1000  Pi={k}=0.3333333333333333  Pi(tail)=0.1111111111111111  Pr(tail)=111/1000
  k=3: Pr={k}=74/1000  Pi={k}=0.1111111111111111  Pi(tail)=3.7037037037037035e-2  Pr(tail)=37/1000
  k=4: Pr={k}=25/1000  Pi={k}=3.7037037037037035e-2  Pi(tail)=1.2345679012345678e-2  Pr(tail)=12/1000

Вывод: {k} возможнее хвоста {k+1,...} при q>1,
хотя вероятности тоже упорядочены (Pr({k}) > Pr(хвоста) для q>1).

--- Пороговый переход: когда возможность становитс

## Связь с субъективным моделированием

Математический аппарат теории возможностей **дословно совпадает** с аппаратом
правдоподобия/доверия субъективного моделирования: те же дуальные шкалы $L,\bar L$, те же меры
$\Pi=\mathrm{Pl}$ и $N=\mathrm{Bel}$ и те же интегралы Сугено. Это **одна** математическая
конструкция.

Различие — только в **интерпретации**:

| | Теория возможностей | Субъективное моделирование |
|--|--|--|
| Источник $\tau,\bar\tau$ | частоты, эмпирика, объективная реальность | интуиция эксперта (эмпирика — лишь один из вариантов) |
| Роль | альтернатива вероятности | способ выражения знания эксперта |
| Приоритет | согласие с наблюдаемыми частотами | суждение эксперта |

Полное изложение мер $\mathrm{Pl}$/$\mathrm{Bel}$, их интегралов и пример экспертной
диагностики двигателя — в ноутбуке
**[SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)**.

![Переход: вероятность -> субъективная модель](../diagrams/misc/uncertainty_to_subj.svg)

## Сводка

Рассмотрены 9 структур описания неопределённости: неопределённость как функтор, монада дискретных распределений, монада Гири и марковские ядра, нечёткая математика, модальные логики и фреймы Крипке, стохастическое программирование с байесовскими сетями, марковские цепи в категории Клейсли, категорная картина стохастических матриц и теории меры, и **теория возможностей Пытьева** как решёточная альтернатива вероятности.

| № | Тема | Ключевая идея |
|---|------|---------------|
| 1 | Функторы неопределённости | Maybe, Either, список, Validation |
| 2 | Монада дискретных распределений | Байесовское обновление, дискретные вероятности |
| 3 | Монада Гири | Марковские ядра, вероятностные меры на пространстве |
| 4 | Нечёткая математика | Треугольные нормы, нечёткие множества, решётки |
| 5 | Модальные логики | Операторы Box и Diamond, фреймы Крипке |
| 6 | Стохастическое программирование | Байесовские сети, предметно-ориентированный язык |
| 7 | Марковские цепи | Цепи в категории Клейсли, стационарные распределения |
| 8 | Стохастические матрицы и теория меры | Энтропия, дивергенция Кульбака-Лейблера |
| 9 | Теория возможностей Пытьева | Pi=sup tau, N=inf tauBar; альтернатива вероятности |

---

**← Предыдущий:** [Comonads](Comonads.ipynb) | **→ Следующий:** [AlgebrasCoalgebras](AlgebrasCoalgebras.ipynb)